In [4]:
EVAL_DATASET = [ 
    # Constitutional Fundamentals 
    
    { 'question': 'Which article guarantees right to equality before law?', 
    'ground_truth': 'Article 14 guarantees the right to equality before law.' }, 
    
    { 'question': 'What are the fundamental rights under the Indian Constitution?', 
    'ground_truth': 'Part III Articles 12-35 guarantee six fundamental rights: equality (Art 14-18), freedom (Art 19-22), against exploitation (Art 23-24), religion (Art 25-28), culture/education (Art 29-30), constitutional remedies (Art 32).' }, 
    
    { 'question': 'Can a state law override central law in India?', 
    'ground_truth': 'No. Article 254 states that in case of inconsistency between central and state law on a Concurrent List subject, central law prevails.' }, 
    
    { 'question': 'What is the right against self-incrimination?', 
    'ground_truth': 'Article 20(3) states no person accused of an offence shall be compelled to be a witness against himself.' }, 
    
    { 'question': 'What is the punishment for murder under IPC?',
    'ground_truth': 'Section 302 of IPC provides that whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine.' }, 
    
    { 'question': 'What is habeas corpus?', 
    'ground_truth': 'Habeas corpus is a writ issued under Article 226/32 directing production of a detained person before court to determine if detention is lawful.' }, 
    
    { 'question': 'What is the period of limitation for a civil suit in India?', 
    'ground_truth': 'Under the Limitation Act 1963, the general period for civil suits is 3 years from the date when the right to sue accrues.' }, 
    
    { 'question': 'What does Article 21 of the Constitution guarantee?', 
    'ground_truth': 'Article 21 guarantees the right to life and personal liberty — no person shall be deprived of life or personal liberty except according to procedure established by law.' }, 
    ]

## run RAGAS Evaluation

In [ ]:
# ── Cell 3: RAGAS Evaluation (Rate-Limit Safe) ────────────────────────────────

import os
import time
import pandas as pd
from dotenv import load_dotenv
from groq import RateLimitError

from ragas import evaluate, EvaluationDataset, SingleTurnSample, RunConfig
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall, AnswerCorrectness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

import sys
sys.path.append(r'../backend')
from agents.legal_graph import run_legal_rag

load_dotenv()

# ── Step 1: Use a small model for RAGAS judge ─────────────────────────────────
# llama3-8b-8192 costs ~6x fewer tokens than llama-3.3-70b-versatile.
# For evaluation judging (faithfulness, relevancy scores), 8B is accurate enough.
# Keep your 70B model only for the actual RAG answer generation in legal_graph.py.
eval_llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,
    max_tokens=512,           # evaluation responses are short — cap them hard
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

judge_llm = LangchainLLMWrapper(eval_llm)

ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
)

# ── Step 2: Metrics — defined AFTER judge_llm exists ─────────────────────────
METRICS = [
    Faithfulness(llm=judge_llm),
    AnswerRelevancy(llm=judge_llm, embeddings=ragas_emb),
    ContextRecall(llm=judge_llm),
    AnswerCorrectness(llm=judge_llm),
]

# Force-set embeddings to guarantee no OpenAI fallback
for m in METRICS:
    if hasattr(m, "embeddings") and m.embeddings is None:
        m.embeddings = ragas_emb

# ── Step 3: RunConfig — valid parameters only ────────────────────────────────
# max_workers=1 forces serial evaluation (no parallel LLM calls)
# timeout=120 gives each metric call 2 minutes before giving up
run_cfg = RunConfig(
    timeout=120,
    max_workers=1,
)

# ── Step 4: Inference loop with retry + delay ─────────────────────────────────
# DELAY_BETWEEN_QUESTIONS: pause between each RAG call to spread token usage.
# At 8 questions × ~1,800 tokens each = ~14,400 tokens for inference alone.
# 10 seconds between questions keeps you well under burst limits.
DELAY_BETWEEN_QUESTIONS = 10   # seconds — increase to 20 if you still hit limits

def run_with_retry(question: str, max_retries: int = 5) -> dict:
    """
    Runs run_legal_rag with exponential backoff on RateLimitError.
    Waits exactly as long as Groq tells us to wait (parsed from error message),
    then retries automatically.
    """
    for attempt in range(max_retries):
        try:
            return run_legal_rag(question)
        except RateLimitError as e:
            error_msg = str(e)

            # Parse the exact wait time from Groq's error message
            # e.g. "Please try again in 2m11.328s"
            import re
            match = re.search(
                r"try again in (?:(\d+)m)?(\d+(?:\.\d+)?)s",
                error_msg
            )
            if match:
                minutes = int(match.group(1) or 0)
                seconds = float(match.group(2) or 0)
                wait    = minutes * 60 + seconds + 5   # +5s buffer
            else:
                wait = 60 * (attempt + 1)              # fallback: 60s, 120s, 180s...

            print(f"\n  ⏳ Rate limit hit (attempt {attempt+1}/{max_retries})")
            print(f"  Groq says wait {wait:.0f}s — pausing now...")

            # Countdown so you can see progress in the notebook
            for remaining in range(int(wait), 0, -10):
                print(f"  Resuming in {remaining}s...", end="\r")
                time.sleep(min(10, remaining))

            print(f"\n  🔄 Retrying question after {wait:.0f}s wait...")

    raise RuntimeError(f"Failed after {max_retries} retries for: {question[:60]}")


print(f"Starting inference for {len(EVAL_DATASET)} questions")
print(f"Delay between questions: {DELAY_BETWEEN_QUESTIONS}s")
print(f"Estimated minimum time: {len(EVAL_DATASET) * DELAY_BETWEEN_QUESTIONS}s\n")

samples = []

for i, sample in enumerate(EVAL_DATASET):
    q = sample["question"]
    print(f"[{i+1}/{len(EVAL_DATASET)}] {q[:65]}...")

    # Run with automatic retry on rate limit
    result = run_with_retry(q)

    answer = result.get("answer") or "No answer generated"
    docs   = result.get("final_docs") or []
    ctxs   = [d.page_content for d in docs] if docs else ["No context retrieved"]

    samples.append(SingleTurnSample(
        user_input=q,
        response=answer,
        retrieved_contexts=ctxs,
        reference=sample["ground_truth"],
    ))

    print(f"Done — strategy: {result.get('strategy', 'N/A')}")

    # Pause between questions — skip after the last one
    if i < len(EVAL_DATASET) - 1:
        print(f"  💤 Waiting {DELAY_BETWEEN_QUESTIONS}s before next question...")
        time.sleep(DELAY_BETWEEN_QUESTIONS)

print(f"\nInference complete — {len(samples)} samples collected")

# ── Step 5: RAGAS evaluation ──────────────────────────────────────────────────
# RAGAS itself makes ~2-4 LLM calls per sample per metric.
# 8 samples × 4 metrics × ~3 calls = ~96 judge calls.
# max_workers=1 spaces these out serially to avoid burst rate limits.

print("\nStarting RAGAS evaluation (this will take several minutes)...")
print("max_workers=1 means metrics are scored one at a time — expected behaviour.\n")

eval_dataset = EvaluationDataset(samples=samples)

results = evaluate(
    dataset=eval_dataset,
    metrics=METRICS,
    llm=judge_llm,
    embeddings=ragas_emb,
    run_config=run_cfg,
    show_progress=True,
)

# ── Step 6: Results ───────────────────────────────────────────────────────────
df   = results.to_pandas()
COLS = [c for c in
        ["faithfulness", "answer_relevancy", "context_recall", "answer_correctness"]
        if c in df.columns]

print("\n===== AGGREGATE SCORES =====")
print(df[COLS].agg(["mean", "min", "max"]).round(3).to_string())

print("\n===== PER-QUESTION SCORES =====")
display_df = df[COLS].copy()
display_df.insert(0, "question", [s.user_input[:50] for s in samples])
print(display_df.to_string(index=False))

df.to_csv("evaluation_results.csv", index=False)
print("\nSaved to evaluation_results.csv")

python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 19


Starting inference for 8 questions
Delay between questions: 10s
Estimated minimum time: 80s

[1/8] Which article guarantees right to equality before law?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[2/8] What are the fundamental rights under the Indian Constitution?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[3/8] Can a state law override central law in India?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[4/8] What is the right against self-incrimination?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[5/8] What is the punishment for murder under IPC?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[6/8] What is habeas corpus?...
 [Strategist] Route: LEGAL
  ✅ Done — strategy: LEGAL
  💤 Waiting 10s before next question...
[7/8] What

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

Exception raised in Job[19]: TimeoutError()
Exception raised in Job[3]: BadRequestError(Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}})
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[16]: TimeoutError()


KeyboardInterrupt: 

Exception raised in Job[30]: BadRequestError(Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}})
Exception raised in Job[9]: BadRequestError(Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}})
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[28]: BadRequestError(Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use 